In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import uproot
import awkward as ak

In [ ]:
def direction_theta_phi(dx, dy, dz, degrees=True):
    rho = np.hypot(dx, dy)
    r = np.hypot(rho, dz)

    theta = ak.where(r > 0, np.arctan2(rho, -dz), np.nan)
    phi   = np.arctan2(dy, dx) 
    phi   = (phi + 2*np.pi) % (2*np.pi)

    if degrees:
        theta = np.degrees(theta)
        phi = np.degrees(phi)

    return r, theta, phi


In [ ]:
datapath = '/proj/internal_group/cup_mc/users/jeewon/AMoRE/Stage1/02_muon/01_data/g4v10/abs_length_500cm'
nruns = 1000
for i in range(nruns):
    print(f"Processing run {i}/{nruns}", end='\r')
    file = uproot.open(datapath + f"/repro_muon/repro_run{i}.root")
    tree = file["prod"]

    evtinfo = tree['evtinfo'].array(library='np')
    v0x, v0y, v0z = tree['vertex.x0'].array(library='ak'), tree['vertex.y0'].array(library='ak'), tree['vertex.z0'].array(library='ak')
    vname = tree['vertex.pname'].array(library='ak')
    cavern_vx, cavern_vy, cavern_vz = tree['cavern.x'].array(library='ak'), tree['cavern.y'].array(library='ak'), tree['cavern.z'].array(library='ak')
    cavern_pname = tree['cavern.pname'].array(library='ak')
    vx, vy, vz = np.array([x[0] if len(x) > 0 else np.nan for x in cavern_vx]), np.array([y[0] if len(y) > 0 else np.nan for y in cavern_vy]), np.array([z[0] if len(z) > 0 else np.nan for z in cavern_vz])
    pname = np.array([x[0] if len(x) > 0 else np.nan for x in cavern_pname])

    dx, dy, dz = vx - v0x, vy - v0y, vz - v0z
    r, theta, phi = direction_theta_phi(dx, dy, dz, degrees=True)
    psveto_edep = tree['psveto.edep'].array(library='ak')
    psveto_id = tree['psveto.id'].array(library='ak')
    npsveto = evtinfo['npsvetohit']

    trim_file = uproot.open(datapath + f"/trim_muon/run{i}.root")
    trim_tree = trim_file["tree"]

    ps_x, ps_y, ps_z, ps_pname = trim_tree['ps_x'].array(library='np'), trim_tree['ps_y'].array(library='np'), trim_tree['ps_z'].array(library='np'), trim_tree['ps_pname'].array(library='np')
    nps = trim_tree['npshit'].array(library='np')
    psf = trim_tree['psfront'].array(library='np')
    psb = trim_tree['psback'].array(library='np')
    psl = trim_tree['psleft'].array(library='np')
    psr = trim_tree['psright'].array(library='np')
    psbot = trim_tree['psbottom'].array(library='np')

    # ps coincidence mask
    fb = (psf == True) & (psb == True) & (psl == False) & (psr == False) & (psbot == False)
    fl = (psf == True) & (psl == True) & (psb == False) & (psr == False) & (psbot == False)
    fr = (psf == True) & (psr == True) & (psb == False) & (psl == False) & (psbot == False)
    bl = (psb == True) & (psl == True) & (psf == False) & (psr == False) & (psbot == False)
    br = (psb == True) & (psr == True) & (psf == False) & (psl == False) & (psbot == False)
    fbot = (psf == True) & (psbot == True) & (psb == False) & (psl == False) & (psr == False)
    bbot = (psb == True) & (psbot == True) & (psf == False) & (psl == False) & (psr == False)
    lbot = (psl == True) & (psbot == True) & (psf == False) & (psb == False) & (psr == False)
    rbot = (psr == True) & (psbot == True) & (psf == False) & (psb == False) & (psl == False)
    ps_coincidence_mask = (fb | fl | fr | bl | br | fbot | bbot | lbot | rbot)

    r_ps_coincidence = r[ps_coincidence_mask]
    theta_ps_coincidence = theta[ps_coincidence_mask]
    phi_ps_coincidence = phi[ps_coincidence_mask]
    psveto_edep_ps_coincidence = psveto_edep[ps_coincidence_mask]
    psveto_id_ps_coincidence = psveto_id[ps_coincidence_mask]

    outfile = uproot.recreate(f'./ps_coincidence/angle_ps_coincidence_run{i}.root')
    outfile['tree'] = {'theta': theta_ps_coincidence, 'phi': phi_ps_coincidence, 'r': r_ps_coincidence, 'psveto_edep': psveto_edep_ps_coincidence, 'psveto_id': psveto_id_ps_coincidence}


    



/tmp/ipykernel_3865193/2783570411.py:54: FutureWarning: Starting in version 5.7.0, Uproot will default to writing RNTuples instead of TTrees. You will need to use `mktree` to explicitly create a TTree. This can be done by changing `file['tree_name'] = data` to `file.mktree('tree_name', data)`. Please update your code accordingly.
  outfile['tree'] = {'theta': theta_ps_coincidence, 'phi': phi_ps_coincidence, 'r': r_ps_coincidence, 'psveto_edep': psveto_edep_ps_coincidence, 'psveto_id': psveto_id_ps_coincidence}
